# NBA Assists Model — V1 | Exploratory and Probabilistic Pipeline

Adapted from your NBA rebounds/points notebooks. **Target: `assists`**. Feature groups include assists, assist-to-turnover ratio, assist percentage, team effective FG%, opponent defensive profiles, playing time, and teammate scoring opportunities. All game outcomes are shifted before modelling.

**Caution:** `assistPercentage` and `assistToTurnoverRatio` are player-game metrics; never use their current-game values. Team eFG% is aggregated from team FGM, 3PM and FGA. Opponent defensive profiles are built from actual *opposing players' assists against that defense* in prior games. Historical pregame lineup status is not available from final box scores.

This notebook is provided as runnable code but has not been trained against your Kaggle dataset here.

## 1. Load data (original source)

In [ ]:
##Import Data

import polars as pl
import numpy as np
import kagglehub
import datetime as dt
from kagglehub import KaggleDatasetAdapter



lf_player_stats = kagglehub.dataset_load(
    KaggleDatasetAdapter.POLARS,
    "eoinamoore/historical-nba-data-and-player-box-scores",
    "PlayerStatisticsExtended.csv",
    polars_kwargs={
        "schema_overrides": {
            "seriesGameNumber": pl.String,
            "numMinutes": pl.String,

            # Basic box score statistics
            "points": pl.Float64,
            "reboundsDefensive": pl.Float64,
            "reboundsOffensive": pl.Float64,
            "reboundsTotal": pl.Float64,
            "foulsPersonal": pl.Float64,
            "assists": pl.Float64,
            "steals": pl.Float64,
            "blocks": pl.Float64,
            "turnovers": pl.Float64,

            # Shooting
            "fieldGoals": pl.Float64,
            "fieldGoalsAttempted": pl.Float64,
            "fieldGoalsMade": pl.Float64,
            "fieldGoalsPercentage": pl.Float64,

            "threePointersMade": pl.Float64,
            "threePointersAttempted": pl.Float64,
            "threePointersPercentage": pl.Float64,

            "freeThrowsMade": pl.Float64,
            "freeThrowsAttempted": pl.Float64,
            "freeThrowsPercentage": pl.Float64,

            "plusMinusPoints": pl.Float64,

            # Advanced statistics
            "offensiveRating": pl.Float64,
            "defensiveRating": pl.Float64,
            "assistPercentage": pl.Float64,
            "assistToTurnoverRatio": pl.Float64,
            "assistRatio": pl.Float64,
            "offensiveReboundPercentage": pl.Float64,
            "defensiveReboundPercentage": pl.Float64,
            "reboundPercentage": pl.Float64,
            "estimatedTurnoverPercentage": pl.Float64,
            "effectiveFieldGoalPercentage": pl.Float64,
            "usagePercentage": pl.Float64,
            "pace": pl.Float64,
            "pacePer40": pl.Float64,

            # Percentage of team production
            "percentTeamOffensiveRebounds": pl.Float64,
            "percentTeamDefensiveRebounds": pl.Float64,
            "percentTeamBlocks": pl.Float64,
            "percentTeamPoints": pl.Float64,
        }
    }
)





lf_team_stats = kagglehub.dataset_load(
  KaggleDatasetAdapter.POLARS,
  "eoinamoore/historical-nba-data-and-player-box-scores",
  "TeamStatisticsExtended.csv"

  # Provide any additional arguments like
  # sql_query, polars_frame_type, or
  # polars_kwargs.
  # See the documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpolars
)


## 2. Clean and standardise historical player box scores (original cleaning)

In [ ]:
## ============================================================
# BASIC CLEANING OF DATA
# ============================================================


# ============================================================
# 1. CONVERT numMinutes FROM MM:SS TO DECIMAL MINUTES
# ============================================================

lf_player_stats = lf_player_stats.with_columns(
    pl.when(
        pl.col("numMinutes").is_not_null()
        & pl.col("numMinutes").str.contains(":")
    )
    .then(
        pl.col("numMinutes")
        .str.split_exact(":", 1)
        .struct.field("field_0")
        .cast(pl.Float64)
        +
        (
            pl.col("numMinutes")
            .str.split_exact(":", 1)
            .struct.field("field_1")
            .cast(pl.Float64)
            / 60
        )
    )
    .otherwise(
        pl.col("numMinutes").cast(pl.Float64, strict=False)
    )
    .alias("numMinutes")
)


# ============================================================
# 2. SELECT RELEVANT COLUMNS
# ============================================================

lf_player_stats = lf_player_stats.select(

    # --------------------------------------------------------
    # Identifiers
    # --------------------------------------------------------
    "personId",
    "gameId",
    "gameDateTimeEst",

    # --------------------------------------------------------
    # Team / opponent
    # --------------------------------------------------------
    "playerteamCity",
    "playerteamName",
    "playerteamId",

    "opponentteamCity",
    "opponentteamName",
    "opponentteamId",

    "home",

    # --------------------------------------------------------
    # Basic player box score
    # --------------------------------------------------------
    "numMinutes",
    "points",
    "assists",
    "blocks",
    "steals",

    "fieldGoalsAttempted",
    "fieldGoalsMade",
    "fieldGoalsPercentage",

    "threePointersAttempted",
    "threePointersMade",
    "threePointersPercentage",

    "freeThrowsAttempted",
    "freeThrowsMade",
    "freeThrowsPercentage",

    "reboundsDefensive",
    "reboundsOffensive",
    "reboundsTotal",

    "turnovers",
    "foulsPersonal",
    "plusMinusPoints",

    # --------------------------------------------------------
    # Advanced player statistics
    # --------------------------------------------------------
    "offensiveRating",
    "defensiveRating",

    "assistPercentage",
    "assistToTurnoverRatio",
    "assistRatio",

    "offensiveReboundPercentage",
    "defensiveReboundPercentage",
    "reboundPercentage",

    "estimatedTurnoverPercentage",
    "effectiveFieldGoalPercentage",

    "usagePercentage",

    "pace",
    "pacePer40",

    # --------------------------------------------------------
    # Percentage of team production
    # --------------------------------------------------------
    "percentTeamOffensiveRebounds",
    "percentTeamDefensiveRebounds",
    "percentTeamBlocks",
    "percentTeamPoints",

    # --------------------------------------------------------
    # Player / game context
    # --------------------------------------------------------
    "startingPosition",
    "gameType",
    "comment"
)


# ============================================================
# 3. CONVERT GAME DATE
# ============================================================

lf_player_stats = (
    lf_player_stats
    .with_columns(
        pl.col("gameDateTimeEst")
        .str.to_datetime("%Y-%m-%d %H:%M:%S")
        .dt.date()
    )
    .filter(
        pl.col("gameDateTimeEst") >= dt.date(1996, 11, 1)
    )
)


# ============================================================
# 4. CREATE STARTER INDICATOR
# ============================================================

lf_player_stats = lf_player_stats.with_columns(
    (
        pl.col("startingPosition").is_not_null()
        &
        (pl.col("startingPosition") != "")
    )
    .cast(pl.Int8)
    .alias("is_starter")
)


# ============================================================
# 5. CREATE PRE-GAME AVAILABILITY INDICATOR
# ============================================================
# Temporary historical proxy until NBA lineup rosterStatus
# data is integrated.
#
# 1 = Available
# 0 = Unavailable due to injury / illness etc.
# ============================================================

lf_player_stats = lf_player_stats.with_columns(

    pl.when(
        pl.col("comment")
        .fill_null("")
        .str.to_lowercase()
        .str.contains(
            r"injur|illness|sick|concussion|sprain|strain|soreness|"
            r"fracture|broken|surgery|rehab|recovery|contusion|"
            r"tendinitis|tendonitis|inflammation|infection|"
            r"back spasms|personal reasons"
        )
    )
    .then(0)
    .otherwise(1)
    .cast(pl.Int8)
    .alias("pregame_available")
)


# ============================================================
# 6. KEEP REGULAR-SEASON GAMES ONLY
# ============================================================

lf_player_stats = lf_player_stats.filter(
    ~pl.col("gameType").is_in([
        "Pre Season",
        "Post Season",
        "All-Star Game",
        "Preseason",
        "Playoffs"
    ])
)

## 3. Engineer strictly pregame assist, team shooting and opponent defensive features

In [ ]:
import polars as pl

ALPHA = 0.60
WINDOWS = (5, 10)

# Explicitly sort by game date and game ID for deterministic history.
def history(lf, metrics, entity, sort_keys, windows=WINDOWS):
    lf = lf.sort(sort_keys)
    lf = lf.with_columns([
        pl.col(c).shift(1).over(entity).alias(f"{c}_lag1")
        for c in metrics
    ])
    lf = lf.with_columns([
        pl.col(c).shift(1).ewm_mean(
            alpha=ALPHA, adjust=False, ignore_nulls=True
        ).over(entity).alias(f"{c}_ewma")
        for c in metrics
    ])
    for w in windows:
        lf = lf.with_columns([
            pl.col(c).shift(1).rolling_mean(
                window_size=w, min_samples=1
            ).over(entity).alias(f"{c}_avg{w}")
            for c in metrics
        ])
    return lf

def safe_ratio(n, d, alias):
    return pl.when(pl.col(d) > 0).then(
        pl.col(n) / pl.col(d)
    ).otherwise(None).alias(alias)

# Only completed, played box-score rows for aggregating actual game stats.
lf_played = lf_player_stats.filter(
    pl.col("numMinutes").is_not_null()
    & (pl.col("numMinutes") > 0)
    & pl.col("assists").is_not_null()
    & pl.col("fieldGoalsAttempted").is_not_null()
)

# TEAM GAME TOTALS — sum player shooting and assists; team pace is approximate.
TEAM_SUM = [
    "assists", "turnovers", "fieldGoalsAttempted", "fieldGoalsMade",
    "threePointersMade", "threePointersAttempted", "points",
]
lf_team_actual = (
    lf_played.group_by([
        "gameId", "gameDateTimeEst", "playerteamId", "opponentteamId"
    ]).agg(
        [pl.col(c).sum().alias(f"team_{c}") for c in TEAM_SUM]
        + [pl.col("pace").mean().alias("team_pace")]
    )
)
lf_team_actual = lf_team_actual.with_columns([
    safe_ratio("team_fieldGoalsMade", "team_fieldGoalsAttempted", "team_fg_pct"),
    (
        pl.when(pl.col("team_fieldGoalsAttempted") > 0)
        .then((pl.col("team_fieldGoalsMade") + 0.5 * pl.col("team_threePointersMade")) /
              pl.col("team_fieldGoalsAttempted"))
        .otherwise(None)
    ).alias("team_efg_pct"),
    safe_ratio("team_assists", "team_fieldGoalsMade", "team_assist_to_fgm"),
    safe_ratio("team_threePointersMade", "team_threePointersAttempted", "team_3p_pct"),
])

# DEFENSIVE PROFILE: assists and shooting ALLOWED by the defense.
# For each team-game row, take the OPPONENT's actual team offense.
opp_cols = [
    "team_assists", "team_fieldGoalsMade", "team_fieldGoalsAttempted",
    "team_threePointersMade", "team_threePointersAttempted", "team_points",
    "team_efg_pct", "team_assist_to_fgm", "team_pace",
]
lf_allowed = lf_team_actual.select(
    ["gameId", pl.col("playerteamId").alias("opponentteamId")]
    + [pl.col(c).alias("allowed_" + c.removeprefix("team_")) for c in opp_cols]
)
lf_team_actual = lf_team_actual.join(
    lf_allowed, on=["gameId", "opponentteamId"], how="left"
)

TEAM_METRICS = [
    "team_assists", "team_turnovers", "team_fieldGoalsMade",
    "team_fieldGoalsAttempted", "team_threePointersMade",
    "team_threePointersAttempted", "team_points", "team_pace",
    "team_fg_pct", "team_efg_pct", "team_assist_to_fgm", "team_3p_pct",
    "allowed_assists", "allowed_fieldGoalsMade", "allowed_fieldGoalsAttempted",
    "allowed_threePointersMade", "allowed_threePointersAttempted",
    "allowed_points", "allowed_efg_pct", "allowed_assist_to_fgm",
    "allowed_pace",
]

# Strictly lagged history for every completed team game.
lf_team_history = history(
    lf_team_actual, TEAM_METRICS, "playerteamId",
    ["playerteamId", "gameDateTimeEst", "gameId"]
)
TEAM_HIST = [f"{c}_{suffix}" for c in TEAM_METRICS
             for suffix in ("lag1", "ewma", "avg5", "avg10")]

# For future scheduled games, construct inclusive snapshots from the
# most recent completed team game, with a STRICT earlier-date join.
lf_snapshots = lf_team_actual.sort(
    ["playerteamId", "gameDateTimeEst", "gameId"]
)
lf_snapshots = lf_snapshots.with_columns([
    pl.col(c).ewm_mean(alpha=ALPHA, adjust=False, ignore_nulls=True)
    .over("playerteamId").alias(f"{c}_ewma") for c in TEAM_METRICS
])
for w in WINDOWS:
    lf_snapshots = lf_snapshots.with_columns([
        pl.col(c).rolling_mean(window_size=w, min_samples=1)
        .over("playerteamId").alias(f"{c}_avg{w}") for c in TEAM_METRICS
    ])
lf_snapshots = lf_snapshots.with_columns([
    pl.col(c).alias(f"{c}_lag1") for c in TEAM_METRICS
]).select(["playerteamId", "gameDateTimeEst"] + TEAM_HIST)

lf_schedule = lf_player_stats.select([
    "gameId", "gameDateTimeEst", "playerteamId", "opponentteamId"
]).unique()
lf_asof = (
    lf_schedule.sort(["gameDateTimeEst", "playerteamId"])
    .join_asof(
        lf_snapshots.sort(["gameDateTimeEst", "playerteamId"]),
        on="gameDateTimeEst", by="playerteamId", strategy="backward",
        allow_exact_matches=False,
    )
)
# Exact lagged features for completed games, as-of for scheduled games.
lf_team_feature = lf_asof.join(
    lf_team_history.select(["gameId", "playerteamId"] + TEAM_HIST),
    on=["gameId", "playerteamId"], how="left", suffix="_exact"
)
lf_team_feature = lf_team_feature.with_columns([
    pl.coalesce(pl.col(f"{c}_exact"), pl.col(c)).alias(c)
    for c in TEAM_HIST
]).drop([f"{c}_exact" for c in TEAM_HIST])

# Own-team offense and upcoming opponent's defense are separate joins.
lf_own = lf_team_feature.select(["gameId", "playerteamId"] + TEAM_HIST)
DEFENSE_HIST = [c for c in TEAM_HIST if c.startswith("allowed_")]
lf_opp = lf_team_feature.select(
    ["gameId", pl.col("playerteamId").alias("opponentteamId")]
    + [pl.col(c).alias(f"opponent_{c}") for c in DEFENSE_HIST]
)
lf_player_stats = (
    lf_player_stats
    .join(lf_own, on=["gameId", "playerteamId"], how="left")
    .join(lf_opp, on=["gameId", "opponentteamId"], how="left")
)

# PLAYER FORM. Source assistPercentage and assistToTurnoverRatio are
# historical player-game advanced statistics. Guard null/invalid ratios.
lf_player_stats = lf_player_stats.with_columns([
    (
        pl.when(pl.col("numMinutes") > 0)
        .then(pl.col("assists") * 36 / pl.col("numMinutes"))
        .otherwise(None)
    ).alias("assists_per36"),
])

PLAYER_METRICS = [
    "assists", "assistToTurnoverRatio", "assistPercentage",
    "numMinutes", "assists_per36", "turnovers", "usagePercentage",
    "pace", "points", "fieldGoalsAttempted", "fieldGoalsMade",
]
lf_player_stats = history(
    lf_player_stats, PLAYER_METRICS, "personId",
    ["personId", "gameDateTimeEst", "gameId"]
)
lf_player_stats = lf_player_stats.with_columns([
    (pl.col("assists_avg5") - pl.col("assists_avg10"))
    .alias("assists_trend_5v10"),
    (pl.col("assistPercentage_avg5") - pl.col("assistPercentage_avg10"))
    .alias("assist_pct_trend_5v10"),
    (pl.col("numMinutes_avg5") - pl.col("numMinutes_avg10"))
    .alias("minutes_trend_5v10"),
])
print("Assist player, team eFG and opponent defense features defined.")


## 4. Assemble pregame assist feature set

In [ ]:
TARGET = "assists"
PLAYER_FEATURES = [
    f"{c}_{suffix}" for c in PLAYER_METRICS
    for suffix in ("lag1", "ewma", "avg5", "avg10")
] + ["assists_trend_5v10", "assist_pct_trend_5v10", "minutes_trend_5v10"]

# Current team's historical eFG, shooting and assists.
TEAM_FEATURES = [c for c in TEAM_HIST if c.startswith("team_")]
# Opponent's OWN prior assists allowed, shooting allowed and pace.
OPPONENT_FEATURES = [f"opponent_{c}" for c in DEFENSE_HIST]
ASSISTS_FEATURES = PLAYER_FEATURES + TEAM_FEATURES + OPPONENT_FEATURES

ID_COLS = [
    "personId", "gameId", "gameDateTimeEst", "playerteamId",
    "opponentteamId", "home"
]
lf_player_stats_assists = (
    lf_player_stats
    .filter(pl.col(TARGET).is_not_null() & (pl.col("numMinutes") > 0))
    .select(ID_COLS + [TARGET] + ASSISTS_FEATURES)
    .sort(["gameDateTimeEst", "gameId", "personId"])
)
print(f"Target: {TARGET}; {len(ASSISTS_FEATURES)} pregame features")


## 5. Mean model benchmarking — LightGBM, XGBoost, optional H2O

In [ ]:
# ============================================================
# 6.1 IMPORTS AND MODELLING CONFIGURATION
# ============================================================
# Install in your notebook environment if needed:
# %pip install lightgbm xgboost scikit-learn pandas joblib
# Optional: %pip install h2o

import time
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

RANDOM_STATE = 42
RUN_H2O = False             # Set True after installing H2O and Java
RUN_ABLATIONS = True
N_JOBS = 4                  # Adjust for your machine
TEST_FRACTION = 0.15
VALIDATION_FRACTION = 0.15

# Optional: restrict the modelling period, while preserving
# feature history generated from earlier seasons.
MIN_MODEL_DATE = None       # e.g. "2020-10-01"

# ============================================================
# 6.2 MATERIALISE AND VALIDATE FEATURE FRAME
# ============================================================
df = lf_player_stats_assists.collect().to_pandas()

if MIN_MODEL_DATE is not None:
    df = df.loc[
        pd.to_datetime(df["gameDateTimeEst"]) >= pd.Timestamp(MIN_MODEL_DATE)
    ].copy()

df["gameDateTimeEst"] = pd.to_datetime(df["gameDateTimeEst"])
df = df.sort_values(["gameDateTimeEst", "gameId", "personId"]).reset_index(drop=True)

assert not df.empty, "No training rows found."
assert df["assists"].notna().all()
assert (df["assists"] >= 0).all()

# Avoid accidental duplicates caused by source data or joins.
dup = df.duplicated(["personId", "gameId"]).sum()
if dup:
    raise ValueError(f"Found {dup} duplicate player-game rows; inspect joins.")

# Pre-game features only: no current-game outcomes, IDs or minutes.
X = df[ASSISTS_FEATURES].copy()
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan).astype("float32")
y = df["assists"].astype("float32")

# Remove all-null features: they provide no learning signal.
all_null = X.columns[X.isna().all()].tolist()
if all_null:
    warnings.warn(f"Dropping {len(all_null)} entirely-null features: {all_null[:10]}")
    X = X.drop(columns=all_null)

FEATURES_USED = X.columns.tolist()
print(f"Rows: {len(df):,} | Features: {len(FEATURES_USED)}")
print(f"Date range: {df.gameDateTimeEst.min().date()} to {df.gameDateTimeEst.max().date()}")
print(f"Target mean: {y.mean():.3f} | Target median: {y.median():.1f}")


In [ ]:
# ============================================================
# 6.3 CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT
# ============================================================
unique_dates = np.array(sorted(df["gameDateTimeEst"].dt.normalize().unique()))
n_dates = len(unique_dates)
train_end = int(n_dates * (1 - VALIDATION_FRACTION - TEST_FRACTION))
val_end = int(n_dates * (1 - TEST_FRACTION))

if train_end < 30 or val_end <= train_end or val_end >= n_dates:
    raise ValueError("Insufficient distinct game dates for 70/15/15 chronological split.")

train_cutoff = unique_dates[train_end]
test_cutoff = unique_dates[val_end]
dates = df["gameDateTimeEst"].dt.normalize().to_numpy()

train_mask = dates < train_cutoff
val_mask = (dates >= train_cutoff) & (dates < test_cutoff)
test_mask = dates >= test_cutoff

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[val_mask], y.loc[val_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

print("Chronological split:")
for name, mask in [("Train", train_mask), ("Validation", val_mask), ("Test", test_mask)]:
    part = df.loc[mask]
    print(f"  {name:10s}: {len(part):>8,} rows | "
          f"{part.gameDateTimeEst.min().date()} to {part.gameDateTimeEst.max().date()} | "
          f"mean AST {part.assists.mean():.3f}")
print("The test set will not be used for model selection.")


In [ ]:
# ============================================================
# 6.4 METRICS AND SIMPLE PREGAME BASELINES
# ============================================================
def regression_metrics(y_true, pred):
    pred = np.clip(np.asarray(pred, dtype=float), 0, None)
    return {
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
        "R2": r2_score(y_true, pred),
        "Bias_actual_minus_pred": float(np.mean(np.asarray(y_true) - pred)),
    }

results = []
validation_predictions = {}

for name, col in [
    ("EWMA baseline", "assists_ewma"),
    ("Last-5 baseline", "assists_avg5"),
    ("Last-10 baseline", "assists_avg10"),
]:
    if col not in X.columns:
        continue
    # Use train mean for cold-start players; do not learn from validation targets.
    fallback = float(y_train.mean())
    pred = X_val[col].fillna(fallback).to_numpy()
    results.append({"Model": name, **regression_metrics(y_val, pred),
                    "Fit_seconds": 0.0})
    validation_predictions[name] = np.clip(pred, 0, None)

display(pd.DataFrame(results).sort_values("MAE"))


In [ ]:
# ============================================================
# 6.5 TRAIN LIGHTGBM AND XGBOOST
# ============================================================
# Both models handle NaNs natively.
# Conservative initial hyperparameters; tune only after benchmarking.
# We do not use the test set for early stopping or tuning.

MODEL_FACTORIES = {
    "LightGBM": lambda: LGBMRegressor(
        objective="regression",
        n_estimators=650,
        learning_rate=0.04,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=75,
        colsample_bytree=0.85,
        subsample=0.85,
        subsample_freq=1,
        reg_lambda=1.0,
        verbosity=-1,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    ),
    "XGBoost": lambda: XGBRegressor(
        objective="reg:squarederror",
        n_estimators=650,
        learning_rate=0.04,
        max_depth=5,
        min_child_weight=10,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=2.0,
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    ),
}

trained_models = {}
for name, factory in MODEL_FACTORIES.items():
    print(f"Starting {name} ...")
    t0 = time.perf_counter()
    model = factory()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - t0
    pred = np.clip(model.predict(X_val), 0, None)
    validation_predictions[name] = pred
    trained_models[name] = model
    scores = regression_metrics(y_val, pred)
    results.append({"Model": name, **scores, "Fit_seconds": elapsed})
    print(f"Finished {name} | MAE={scores['MAE']:.4f} | "
          f"RMSE={scores['RMSE']:.4f} | R²={scores['R2']:.4f} | "
          f"{elapsed:.1f}s")

validation_results = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)
display(validation_results)


### 6.6 Optional H2O AutoML

H2O AutoML is a separate, optional benchmark. It can be substantially slower and requires Java. We disable its internal random leaderboard holdout and explicitly score the same chronological validation rows used for LightGBM and XGBoost. AutoML uses the training data only; the external validation set remains untouched until evaluation. For a strict time-aware *internal* H2O selection, use a chronological `fold_column` or a separate internal time holdout instead of default random cross-validation. The configuration below uses **no CV** and an internal training-frame leaderboard split only; the external validation/test dates remain separated.

In [ ]:
# ============================================================
# 6.6 OPTIONAL H2O AUTOML
# ============================================================
if RUN_H2O:
    import h2o
    from h2o.automl import H2OAutoML

    h2o.init(max_mem_size="8G")

    # H2O needs a response column and numeric predictors.
    # Keep missing values as NaN; H2O tree models can handle them.
    target_col = "assists"
    train_pd = X_train.astype(float).copy()
    train_pd[target_col] = y_train.to_numpy()
    val_pd = X_val.astype(float).copy()

    hf_train = h2o.H2OFrame(train_pd)
    hf_val = h2o.H2OFrame(val_pd)

    print("Starting H2O AutoML ...")
    t0 = time.perf_counter()
    automl = H2OAutoML(
        max_models=12,
        seed=RANDOM_STATE,
        nfolds=0,
        sort_metric="MAE",
        exclude_algos=["DeepLearning"],
        keep_cross_validation_predictions=False,
    )
    automl.train(
        x=FEATURES_USED,
        y=target_col,
        training_frame=hf_train,
    )
    elapsed = time.perf_counter() - t0
    h2o_pred = np.clip(
        automl.leader.predict(hf_val).as_data_frame().iloc[:, 0].to_numpy(),
        0, None
    )
    validation_predictions["H2O AutoML"] = h2o_pred
    scores = regression_metrics(y_val, h2o_pred)
    results.append({"Model": "H2O AutoML", **scores, "Fit_seconds": elapsed})
    validation_results = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)
    print("H2O leader:", automl.leader.model_id)
    display(automl.leaderboard.head())
    display(validation_results)
else:
    print("H2O skipped (RUN_H2O=False).")


In [ ]:
# ============================================================
# 6.7 FEATURE IMPORTANCE: GAIN AND SPLIT COUNTS
# ============================================================
import matplotlib.pyplot as plt

importance_frames = []
for name, model in trained_models.items():
    if name == "LightGBM":
        gain = model.booster_.feature_importance(importance_type="gain")
        split = model.booster_.feature_importance(importance_type="split")
    else:
        booster = model.get_booster()
        gain_map = booster.get_score(importance_type="total_gain")
        split_map = booster.get_score(importance_type="weight")
        gain = np.array([gain_map.get(f, 0.0) for f in FEATURES_USED])
        split = np.array([split_map.get(f, 0.0) for f in FEATURES_USED])

    imp = pd.DataFrame({
        "Model": name,
        "Feature": FEATURES_USED,
        "Total_gain": gain,
        "Splits": split,
    })
    total = imp["Total_gain"].sum()
    imp["Gain_pct"] = 100 * imp["Total_gain"] / total if total > 0 else 0
    importance_frames.append(imp)
    print(f"\n{name} top 25:")
    display(imp.sort_values("Total_gain", ascending=False).head(25))

    top = imp.nlargest(25, "Total_gain").sort_values("Gain_pct")
    plt.figure(figsize=(11, 9))
    plt.barh(top["Feature"], top["Gain_pct"])
    plt.xlabel("Share of total model split gain (%)")
    plt.title(f"{name}: Top 25 assist predictors")
    plt.tight_layout()
    plt.show()

feature_importance = pd.concat(importance_frames, ignore_index=True)


In [ ]:
# ============================================================
# 6.8 VALIDATION ERROR BY ACTUAL ASSIST BUCKET
# ============================================================
bucket_edges = [-0.1, 2.5, 5.5, 8.5, 11.5, np.inf]
bucket_labels = ["0-2", "3-5", "6-8", "9-11", "12+"]

bucket = pd.cut(y_val, bins=bucket_edges, labels=bucket_labels)
bucket_rows = []
for name, pred in validation_predictions.items():
    tmp = pd.DataFrame({
        "bucket": bucket.to_numpy(),
        "actual": y_val.to_numpy(),
        "pred": np.asarray(pred),
    })
    for group, part in tmp.groupby("bucket", observed=True):
        bucket_rows.append({
            "Model": name, "Actual_AST_bucket": str(group),
            "N": len(part),
            "MAE": np.mean(np.abs(part["actual"] - part["pred"])),
            "Bias_actual_minus_pred": np.mean(part["actual"] - part["pred"]),
        })

bucket_errors = pd.DataFrame(bucket_rows)
display(bucket_errors)


### 6.9 Feature-group ablation

This compares validation MAE when removing entire feature groups. It is more meaningful than eliminating individual correlated features based on gain importance alone. Each ablation retrains LightGBM on the **same** chronological training window.

In [ ]:

# ============================================================
# 6.9 FEATURE GROUP ABLATIONS - NBA ASSISTS MODEL
# ============================================================

ablation_results = []

if RUN_ABLATIONS:

    # --------------------------------------------------------
    # 1. DEFINE FEATURE GROUPS
    # --------------------------------------------------------

    player_features = [
        f for f in FEATURES_USED
        if f in PLAYER_FEATURES
    ]

    team_features = [
        f for f in FEATURES_USED
        if f in TEAM_FEATURES
    ]

    opponent_features = [
        f for f in FEATURES_USED
        if f in OPPONENT_FEATURES
    ]

    # --------------------------------------------------------
    # 2. DEFINE ABLATION EXPERIMENTS
    # --------------------------------------------------------

    groups = {

        # Baseline
        "All features": FEATURES_USED,

        # Feature groups
        "Player only": player_features,

        "Player + own team": [
            f for f in FEATURES_USED
            if f in PLAYER_FEATURES or f in TEAM_FEATURES
        ],

        "Player + opponent": [
            f for f in FEATURES_USED
            if f in PLAYER_FEATURES or f in OPPONENT_FEATURES
        ],

        # ----------------------------------------------------
        # PLAYMAKING FEATURES
        # ----------------------------------------------------

        "Without assist percentage": [
            f for f in FEATURES_USED
            if "assistPercentage" not in f
        ],

        "Without assist-to-turnover ratio": [
            f for f in FEATURES_USED
            if not any(
                token in f
                for token in (
                    "assistToTurnoverRatio",
                    "assistRatio",
                )
            )
        ],

        # ----------------------------------------------------
        # ASSIST HISTORY
        # ----------------------------------------------------

        "Without assists avg10": [
            f for f in FEATURES_USED
            if f != "assists_avg10"
        ],

        "Without assists avg5": [
            f for f in FEATURES_USED
            if f != "assists_avg5"
        ],

        "Without assists EWMA": [
            f for f in FEATURES_USED
            if f != "assists_ewma"
        ],

        "Without assist history": [
            f for f in FEATURES_USED
            if not f.startswith("assists_")
        ],

        # ----------------------------------------------------
        # TEAM SHOOTING ENVIRONMENT
        # ----------------------------------------------------

        "Without team shooting": [
            f for f in FEATURES_USED
            if not (
                f in TEAM_FEATURES
                and any(
                    token in f
                    for token in (
                        "fieldGoals",
                        "threePointers",
                        "freeThrows",
                        "fg_pct",
                        "3p_pct",
                        "ft_pct",
                        "efg",
                        "effectiveFieldGoal",
                    )
                )
            )
        ],

        # ----------------------------------------------------
        # OPPONENT DEFENSIVE PROFILE
        # ----------------------------------------------------

        "Without opponent defense": [
            f for f in FEATURES_USED
            if f not in OPPONENT_FEATURES
        ],

        # ----------------------------------------------------
        # PLAYING OPPORTUNITY
        # ----------------------------------------------------

        "Without minutes features": [
            f for f in FEATURES_USED
            if "numMinutes" not in f
            and "assists_per36" not in f
        ],
    }

    # --------------------------------------------------------
    # 3. TRAIN AND EVALUATE
    # --------------------------------------------------------

    for group_name, cols in groups.items():

        if not cols:
            continue

        print(
            f"Starting: {group_name} "
            f"({len(cols)} features)"
        )

        model = MODEL_FACTORIES["LightGBM"]()

        model.fit(
            X_train[cols],
            y_train
        )

        pred = np.clip(
            model.predict(X_val[cols]),
            0,
            None
        )

        metrics = regression_metrics(
            y_val,
            pred
        )

        ablation_results.append({
            "Feature_set": group_name,
            "N_features": len(cols),
            **metrics
        })

        print(
            f"Completed: {group_name} | "
            f"MAE: {metrics['MAE']:.4f}"
        )

    # --------------------------------------------------------
    # 4. COMPARE RESULTS
    # --------------------------------------------------------

    ablation_df = (
        pd.DataFrame(ablation_results)
        .sort_values("MAE")
        .reset_index(drop=True)
    )

    # Difference relative to all-feature baseline
    baseline_mae = ablation_df.loc[
        ablation_df["Feature_set"] == "All features",
        "MAE"
    ].iloc[0]

    ablation_df["MAE_change_vs_full"] = (
        ablation_df["MAE"] - baseline_mae
    )

    display(ablation_df)


## 7. Final model selection and untouched test evaluation

Select between **LightGBM** and **XGBoost** using validation MAE (and inspect bias/bucket errors). H2O is an exploratory comparator here; if it wins, a separate H2O-specific chronological refit and export should be implemented rather than silently substituting a different model. The selected tree model is then refit on train + validation dates and evaluated once on the held-out test dates.

In [ ]:
# ============================================================
# 7.1 SELECT, REFIT, AND EVALUATE ON TEST
# ============================================================
eligible = validation_results.loc[
    validation_results["Model"].isin(MODEL_FACTORIES)
].sort_values("MAE")
if eligible.empty:
    raise RuntimeError("No LightGBM/XGBoost model available to select.")

BEST_MODEL_NAME = str(eligible.iloc[0]["Model"])
print("Selected mean model:", BEST_MODEL_NAME)

X_trainval = X.loc[train_mask | val_mask]
y_trainval = y.loc[train_mask | val_mask]

final_mean_model = MODEL_FACTORIES[BEST_MODEL_NAME]()
t0 = time.perf_counter()
final_mean_model.fit(X_trainval, y_trainval)
print(f"Final refit took {time.perf_counter() - t0:.1f}s")

test_pred = np.clip(final_mean_model.predict(X_test), 0, None)
test_metrics = regression_metrics(y_test, test_pred)
print("Untouched test results:", test_metrics)

test_diagnostics = df.loc[test_mask, [
    "personId", "gameId", "gameDateTimeEst",
    "playerteamId", "opponentteamId", "assists"
]].copy()
test_diagnostics["predicted_assists"] = test_pred
test_diagnostics["error_actual_minus_pred"] = (
    test_diagnostics["assists"] - test_diagnostics["predicted_assists"]
)
display(test_diagnostics.head(20))


In [ ]:
# ============================================================
# 7.2 EXPORT MEAN MODEL AND METADATA
# ============================================================
import json
import joblib

OUT_DIR = Path("models/nba_assists/v1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(final_mean_model, OUT_DIR / "assists_mean_model.joblib")
feature_importance.to_csv(OUT_DIR / "feature_importance.csv", index=False)
validation_results.to_csv(OUT_DIR / "validation_comparison.csv", index=False)
test_diagnostics.to_csv(OUT_DIR / "test_predictions.csv", index=False)

metadata = {
    "target": "assists",
    "model": BEST_MODEL_NAME,
    "features": FEATURES_USED,
    "alpha_ewma": ALPHA,
    "train_cutoff_exclusive": str(pd.Timestamp(train_cutoff).date()),
    "test_cutoff_inclusive": str(pd.Timestamp(test_cutoff).date()),
    "trainval_rows": int(len(X_trainval)),
    "test_rows": int(len(X_test)),
    "test_metrics": {k: float(v) for k, v in test_metrics.items()},
    "availability_features_enabled": False,
    "notes": "Pregame historical form features only; conditional NB not fitted yet.",
}
(OUT_DIR / "model_config.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)
print("Saved artifacts to:", OUT_DIR.resolve())


## 8. Next modelling phase

Once the best mean model has been selected and its **untouched test** performance reviewed, the next stage is probabilistic rebound modelling:

- Generate chronological out-of-fold mean predictions **within the training period**.
- Fit a global Negative Binomial dispersion baseline, and compare against Poisson.
- If warranted, train a conditional dispersion model using the OOF means.
- Evaluate held-out log-loss, Brier scores and calibration at actual rebound lines (e.g. 4.5, 6.5, 8.5, 10.5).
- Verify data completeness, historical player availability, and pregame inference feature parity before treating model probabilities as actionable.

**Caveat:** This notebook has been assembled and syntax-checked, but the full Kaggle dataset and the ML benchmarks have not been executed here. No performance results are asserted.

## 6. Probabilistic assists — Poisson and Negative Binomial

---
## 9. Probabilistic assist model — Poisson vs Negative Binomial

Use the **existing untuned LightGBM hyperparameters**, avoiding another costly search. Fit dispersion from **expanding-window out-of-fold (OOF)** mean predictions, so the dispersion fit does not see in-sample mean predictions. We benchmark:

1. Poisson (variance = mean)
2. Global NB2 (variance = mean + α × mean²)
3. Bucket-conditioned NB2 (α estimated by OOF predicted-mean bucket, with shrinkage to global α)

**Holdout discipline:** choose the distribution using the original chronological validation set; then rebuild the selected distribution using train + validation only and evaluate it once on the untouched test set. This section intentionally does **not** train a complex second LightGBM dispersion model until these simpler baselines are validated.

**Training target:** played-game assists only. Probabilities are conditional on a player participating; handling late scratches/DNP requires a separate pregame availability component.

In [ ]:
# ============================================================
# 9.1 NEGATIVE BINOMIAL HELPERS AND PRICING
# ============================================================
from scipy.optimize import minimize_scalar
from scipy.stats import nbinom, poisson
from scipy.special import gammaln
import numpy as np
import pandas as pd

EPS = 1e-9
ALPHA_MIN, ALPHA_MAX = 1e-4, 5.0


def nb2_nll(y, mu, alpha):
    """Per-observation NB2 negative log likelihood; α > 0."""
    y = np.asarray(y, dtype=float)
    mu = np.maximum(np.asarray(mu, dtype=float), EPS)
    a = np.maximum(np.asarray(alpha, dtype=float), ALPHA_MIN)
    r = 1.0 / a
    return -(
        gammaln(y + r) - gammaln(r) - gammaln(y + 1)
        + r * (np.log(r) - np.log(r + mu))
        + y * (np.log(mu) - np.log(r + mu))
    )


def fit_global_alpha(y, mu):
    y = np.asarray(y, dtype=float)
    mu = np.maximum(np.asarray(mu, dtype=float), EPS)
    opt = minimize_scalar(
        lambda log_a: float(np.mean(nb2_nll(y, mu, np.exp(log_a)))),
        bounds=(np.log(ALPHA_MIN), np.log(ALPHA_MAX)),
        method="bounded", options={"xatol": 1e-4},
    )
    if not opt.success:
        raise RuntimeError(f"Global dispersion fit failed: {opt.message}")
    return float(np.exp(opt.x))


def fit_bucket_alphas(y, mu, global_alpha, edges=(0, 3, 6, 9, 13, np.inf),
                      min_bucket_n=250, prior_n=500):
    """Fit bucket NB2 alpha; shrink toward global alpha for stability."""
    y = np.asarray(y, dtype=float)
    mu = np.asarray(mu, dtype=float)
    bucket_ids = np.digitize(mu, edges[1:-1], right=False)
    alphas = []
    diagnostics = []
    for k in range(len(edges) - 1):
        idx = bucket_ids == k
        count = int(idx.sum())
        local = fit_global_alpha(y[idx], mu[idx]) if count >= min_bucket_n else global_alpha
        weight = count / (count + prior_n) if count >= min_bucket_n else 0.0
        blended = float(np.exp(weight * np.log(local) + (1 - weight) * np.log(global_alpha)))
        alphas.append(blended)
        diagnostics.append({"bucket": k, "lower": edges[k], "upper": edges[k + 1],
                            "n": count, "raw_alpha": local, "shrunk_alpha": blended})
    return {"edges": list(edges), "alphas": alphas}, pd.DataFrame(diagnostics)


def bucket_alpha(mu, spec):
    mu = np.asarray(mu, dtype=float)
    idx = np.digitize(mu, spec["edges"][1:-1], right=False)
    return np.asarray(spec["alphas"])[idx]


def prop_probabilities(mu, line, distribution="nb", alpha=None):
    """Return win/push/loss probabilities for BOTH sides, incl. integer lines.

    For integer lines, a result equal to the line is a push.
    For half-point lines, push probability is zero.
    """
    mu = np.maximum(np.asarray(mu, dtype=float), EPS)
    line = np.asarray(line, dtype=float)
    mu, line = np.broadcast_arrays(mu, line)
    if np.any(line < 0):
        raise ValueError("Prop lines must be nonnegative")
    floor_line = np.floor(line)
    ceil_line = np.ceil(line)
    if distribution == "poisson":
        cdf = lambda k: poisson.cdf(k, mu)
        sf = lambda k: poisson.sf(k, mu)
        pmf = lambda k: poisson.pmf(k, mu)
    elif distribution == "nb":
        if alpha is None:
            raise ValueError("alpha is required for NB")
        a = np.maximum(np.broadcast_to(np.asarray(alpha, dtype=float), mu.shape), ALPHA_MIN)
        r = 1.0 / a
        p = r / (r + mu)
        cdf = lambda k: nbinom.cdf(k, r, p)
        sf = lambda k: nbinom.sf(k, r, p)
        pmf = lambda k: nbinom.pmf(k, r, p)
    else:
        raise ValueError("distribution must be 'poisson' or 'nb'")
    over = sf(floor_line)   # P(Y > line)
    under = cdf(ceil_line - 1)  # P(Y < line)
    push = np.where(floor_line == line, pmf(line), 0.0)
    return {"over": over, "under": under, "push": push}


def fair_odds(win_probability, push_probability=0):
    """Fair decimal odds with stake returned on push: (1-push)/p(win)."""
    win_probability = np.asarray(win_probability, dtype=float)
    push_probability = np.asarray(push_probability, dtype=float)
    return np.where(win_probability > 0,
                    (1 - push_probability) / np.maximum(win_probability, EPS), np.inf)


def expected_roi(win_probability, push_probability, bookmaker_odds):
    """Per-unit-stake EV: p(win)*(odds-1) - p(loss)."""
    return (np.asarray(win_probability) * (np.asarray(bookmaker_odds) - 1)
            - (1 - np.asarray(win_probability) - np.asarray(push_probability)))


### 9.2 Chronological OOF predictions for dispersion

OOF folds use the earliest portion of the training window as a warm-up and then expanding training windows. The last fold ends before the validation period. Refit the same LightGBM configuration each fold. This is substantially cheaper than hyperparameter search and prevents fitting dispersion on overly optimistic in-sample mean predictions.

In [ ]:
# ============================================================
# 9.2 EXPANDING-WINDOW OOF MEAN PREDICTIONS
# ============================================================
# Uses X, y, df, train_mask, val_mask, test_mask and
# MODEL_FACTORIES from the model-development cells above.

PROB_MODEL_NAME = "LightGBM"
PROB_FEATURES = FEATURES_USED.copy()
# Use the same complete, cleaned feature set as the main regression model.
# This includes player, team and opponent historical predictors.
assert PROB_FEATURES and set(PROB_FEATURES).issubset(X.columns)

N_OOF_FOLDS = 4
OOF_WARMUP_FRACTION = 0.40


def expanding_oof_predictions(X_subset, y_subset, dates_subset,
                              feature_names, n_folds=N_OOF_FOLDS,
                              warmup_fraction=OOF_WARMUP_FRACTION):
    dates = pd.to_datetime(pd.Series(dates_subset).reset_index(drop=True)).dt.normalize()
    distinct = np.array(sorted(dates.unique()))
    warmup = max(20, int(len(distinct) * warmup_fraction))
    if len(distinct) - warmup < n_folds:
        raise ValueError("Insufficient dates for chronological OOF folds")
    boundaries = np.linspace(warmup, len(distinct), n_folds + 1, dtype=int)
    X_local = X_subset[feature_names].reset_index(drop=True)
    y_local = pd.Series(y_subset).reset_index(drop=True)
    preds = np.full(len(X_local), np.nan)
    for fold in range(n_folds):
        lo, hi = boundaries[fold], boundaries[fold + 1]
        train_idx = (dates < distinct[lo]).to_numpy()
        hold_idx = ((dates >= distinct[lo]) &
                    (dates < distinct[hi] if hi < len(distinct) else True)).to_numpy()
        model = MODEL_FACTORIES[PROB_MODEL_NAME]()
        model.fit(X_local.loc[train_idx], y_local.loc[train_idx])
        preds[hold_idx] = np.maximum(model.predict(X_local.loc[hold_idx]), EPS)
        print(f"OOF fold {fold + 1}/{n_folds}: train={train_idx.sum():,} "
              f"holdout={hold_idx.sum():,}")
    valid = np.isfinite(preds)
    return preds, valid

# Build OOF strictly within the training dates.
train_dates = df.loc[train_mask, "gameDateTimeEst"].reset_index(drop=True)
train_oof, train_oof_valid = expanding_oof_predictions(
    X_train, y_train, train_dates, PROB_FEATURES
)
print(f"OOF coverage: {train_oof_valid.sum():,}/{len(train_oof):,}")


In [ ]:
# ============================================================
# 9.3 FIT DISPERSION ON TRAINING OOF PREDICTIONS ONLY
# ============================================================
train_oof_y = y_train.to_numpy()[train_oof_valid]
train_oof_mu = train_oof[train_oof_valid]
alpha_train = fit_global_alpha(train_oof_y, train_oof_mu)
bucket_train, bucket_train_diagnostics = fit_bucket_alphas(
    train_oof_y, train_oof_mu, alpha_train
)
print(f"Training OOF global alpha: {alpha_train:.5f}")
display(bucket_train_diagnostics)

# Refit a mean model on ALL training rows, predict validation.
prob_mean_train = MODEL_FACTORIES[PROB_MODEL_NAME]()
prob_mean_train.fit(X_train[PROB_FEATURES], y_train)
val_mu = np.maximum(prob_mean_train.predict(X_val[PROB_FEATURES]), EPS)


### 9.4 Select the probability distribution using validation

Evaluate both **count negative log likelihood** and binary Brier scores at common half-point assist lines. This separates distribution quality from the original mean-model MAE. The best distribution is selected by validation count NLL; Brier scores provide an additional calibration diagnostic.

In [ ]:
# ============================================================
# 9.4 DISTRIBUTION BENCHMARK ON VALIDATION
# ============================================================
from scipy.stats import poisson

PROP_LINES = np.array([1.5, 2.5, 3.5, 4.5, 5.5, 7.5, 9.5])

def distribution_metrics(y_true, mu, kind, alpha=None, lines=PROP_LINES):
    y_true = np.asarray(y_true, dtype=float)
    mu = np.maximum(np.asarray(mu, dtype=float), EPS)
    if kind == "poisson":
        nll = -poisson.logpmf(y_true, mu)
    else:
        nll = nb2_nll(y_true, mu, alpha)
    rows = []
    for line in lines:
        p = prop_probabilities(mu, line, distribution=kind, alpha=alpha)["over"]
        event = (y_true > line).astype(float)
        rows.append({"line": line, "Brier": float(np.mean((p - event) ** 2)),
                     "Observed_over": float(event.mean()), "Predicted_over": float(p.mean())})
    line_df = pd.DataFrame(rows)
    return {"NLL": float(np.mean(nll)),
            "Mean_Brier": float(line_df["Brier"].mean()),
            "Mean_abs_calibration_gap": float(np.mean(np.abs(
                line_df["Predicted_over"] - line_df["Observed_over"])))}, line_df

val_alpha_bucket = bucket_alpha(val_mu, bucket_train)
val_specs = {
    "Poisson": ("poisson", None),
    "NB global": ("nb", alpha_train),
    "NB bucket": ("nb", val_alpha_bucket),
}
validation_prob_results = []
validation_line_diagnostics = []
for label, (kind, alpha) in val_specs.items():
    scores, per_line = distribution_metrics(y_val, val_mu, kind, alpha)
    validation_prob_results.append({"Distribution": label, **scores})
    per_line.insert(0, "Distribution", label)
    validation_line_diagnostics.append(per_line)

validation_prob_results = pd.DataFrame(validation_prob_results).sort_values("NLL")
validation_line_diagnostics = pd.concat(validation_line_diagnostics, ignore_index=True)
display(validation_prob_results)
display(validation_line_diagnostics)
BEST_DISTRIBUTION = str(validation_prob_results.iloc[0]["Distribution"])
print("Selected distribution by validation NLL:", BEST_DISTRIBUTION)


### 9.5 Refit on train + validation and evaluate the untouched test period

Rebuild the dispersion parameters using chronological OOF predictions from **train + validation**, and refit the mean model on the same period. The test outcomes are not used to select or fit any parameters.

In [ ]:
# ============================================================
# 9.5 FINAL REFIT + TEST EVALUATION
# ============================================================
trainval_mask = train_mask | val_mask
X_tv = X.loc[trainval_mask]
y_tv = y.loc[trainval_mask]
dates_tv = df.loc[trainval_mask, "gameDateTimeEst"].reset_index(drop=True)

tv_oof, tv_valid = expanding_oof_predictions(
    X_tv, y_tv, dates_tv, PROB_FEATURES
)
tv_oof_y = y_tv.to_numpy()[tv_valid]
tv_oof_mu = tv_oof[tv_valid]
alpha_final = fit_global_alpha(tv_oof_y, tv_oof_mu)
bucket_final, bucket_final_diagnostics = fit_bucket_alphas(
    tv_oof_y, tv_oof_mu, alpha_final
)

prob_mean_final = MODEL_FACTORIES[PROB_MODEL_NAME]()
prob_mean_final.fit(X_tv[PROB_FEATURES], y_tv)
test_mu = np.maximum(prob_mean_final.predict(X_test[PROB_FEATURES]), EPS)

def select_distribution(name, mu, global_alpha, bucket_spec):
    if name == "Poisson":
        return "poisson", None
    if name == "NB global":
        return "nb", global_alpha
    if name == "NB bucket":
        return "nb", bucket_alpha(mu, bucket_spec)
    raise ValueError(f"Unknown distribution: {name}")

final_kind, test_alpha = select_distribution(
    BEST_DISTRIBUTION, test_mu, alpha_final, bucket_final
)
selected_test_scores, selected_test_lines = distribution_metrics(
    y_test, test_mu, final_kind, test_alpha
)
print("Selected model test metrics:", selected_test_scores)
print("Global alpha:", alpha_final)
display(selected_test_lines)

# Also compare all three on test for reporting, without reselecting.
test_comparison = []
for label in ("Poisson", "NB global", "NB bucket"):
    kind, alpha = select_distribution(label, test_mu, alpha_final, bucket_final)
    scores, _ = distribution_metrics(y_test, test_mu, kind, alpha)
    test_comparison.append({"Distribution": label, **scores})
display(pd.DataFrame(test_comparison).sort_values("NLL"))


### 9.6 Fair odds and EV for over/under assist props

Use the bookmaker's **actual** line and odds. For half-point lines, there is no push. For integer lines, fair odds and EV correctly account for the possibility of a push (stake returned). These are **model-implied** prices, not market-calibrated probabilities; do not interpret positive estimated EV as guaranteed profitability.

In [ ]:
# ============================================================
# 9.6 EXAMPLE: REBOUND PROBABILITIES AND FAIR PRICES
# ============================================================
example_mu = 5.2
example_line = 5.5
example_bookmaker_over = 1.95
example_bookmaker_under = 1.90

example_kind, example_alpha = select_distribution(
    BEST_DISTRIBUTION,
    np.array([example_mu]),
    alpha_final,
    bucket_final,
)
example_probs = prop_probabilities(
    example_mu, example_line, distribution=example_kind,
    alpha=example_alpha,
)
for side, odds in [("over", example_bookmaker_over),
                   ("under", example_bookmaker_under)]:
    p_win = float(np.asarray(example_probs[side]).item())
    p_push = float(np.asarray(example_probs["push"]).item())
    print(f"{side.upper()} {example_line}: win={p_win:.2%}, "
          f"push={p_push:.2%}, fair_odds={float(fair_odds(p_win, p_push)):.3f}, "
          f"EV={float(expected_roi(p_win, p_push, odds)):.2%}")


### 9.7 Save the probability model and its parameters

The model, feature list, and distribution metadata must travel together. For future daily predictions, regenerate exactly the same pregame feature columns and use the stored distribution selection and dispersion parameters.

In [ ]:
# ============================================================
# 9.7 SAVE PROBABILISTIC REBOUND MODEL
# ============================================================
import json
import joblib
from pathlib import Path

PROB_OUT_DIR = Path("models/nba_assists/v1_probabilistic")
PROB_OUT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(prob_mean_final, PROB_OUT_DIR / "assists_mean_lightgbm.joblib")

prob_config = {
    "target": "assists",
    "mean_model": PROB_MODEL_NAME,
    "features": PROB_FEATURES,
    "distribution": BEST_DISTRIBUTION,
    "nb2_global_alpha": float(alpha_final),
    "nb2_bucket_edges": [float(v) for v in bucket_final["edges"]],
    "nb2_bucket_alphas": [float(v) for v in bucket_final["alphas"]],
    "ewma_alpha": float(ALPHA),
    "test_scores": {k: float(v) for k, v in selected_test_scores.items()},
    "conditioning": "Player participated; DNP probability not modelled",
    "selection_metric": "Validation mean count NLL",
}
(PROB_OUT_DIR / "probability_config.json").write_text(
    json.dumps(prob_config, indent=2), encoding="utf-8"
)
validation_prob_results.to_csv(PROB_OUT_DIR / "validation_distributions.csv", index=False)
validation_line_diagnostics.to_csv(PROB_OUT_DIR / "validation_lines.csv", index=False)
selected_test_lines.to_csv(PROB_OUT_DIR / "test_lines.csv", index=False)
print("Saved probabilistic model to:", PROB_OUT_DIR.resolve())


### Before live use

- Validate pregame feature availability and joins; the feature pipeline must not use same-game box-score data at inference.
- Check line-level calibration on actual offered bookmaker lines and on a recent untouched season; synthetic fixed lines are only a preliminary diagnostic.
- Investigate player availability, minutes uncertainty and the distribution's tail calibration. A simple NB2 model may not adequately capture all assist distributions.
- For a more sophisticated follow-up, use the custom conditional-dispersion LightGBM approach from the original points notebook, but only if it improves held-out count NLL and Brier scores.